In [ ]:

import os
import torch
import sys
import numpy as np
from glob import glob
import cv2
import random
from scipy.io import loadmat
import matplotlib.pyplot as plt
import seaborn as sns

from utils import *
from PIL import Image
from torchvision import transforms
import cv2
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from tensorflow import keras

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

import segmentation_models_pytorch as smp
import pandas as pd

from scipy.stats import mannwhitneyu
import skimage.morphology as morph

# Change the working directory based on data location
all_images = sorted(glob("data/lifeact_3t3/*"))

# Loading the Colormap
colormap = np.array([[0.0 , 0.0, 0.0], # Background - Black
                     [0.0, 0.99, 0.0], # Actin - Green
                     [0.99, 0.0, 0.0], # Focal Adhesions - Red
                     [0.99, 0.99, 0.0], # Lamellipodin - Yellow
                     [0.0, 0.0, 0.99], # Filopodia - Blue
                     ]) 

colormap = colormap * 100
colormap = colormap.astype(np.uint8)


device = "cuda" if torch.cuda.is_available() else "cpu"
model = smp.UnetPlusPlus(
    encoder_name="resnet34",
    encoder_weights="imagenet", 
    in_channels=1,
    classes=5,  
).to(device)

# Load the model from UnetPlusPlus/best_model.pth
model.load_state_dict(torch.load("unetpp_best_yet4/model/best_model.pth"))
model.eval()

transform = transforms.Compose([
    transforms.ToTensor(),
])

def decode_segmentation_masks(mask, colormap, n_classes):
    r = np.zeros_like(mask).astype(np.uint8)
    g = np.zeros_like(mask).astype(np.uint8)
    b = np.zeros_like(mask).astype(np.uint8)
    for l in range(0, n_classes):
        idx = mask == l
        r[idx] = colormap[l, 0]
        g[idx] = colormap[l, 1]
        b[idx] = colormap[l, 2]
    rgb = np.stack([r, g, b], axis=2)
    return rgb

def detect_stress_fibers(image, mask, cell_area):
    mask_no_boundary = remove_cell_boundary(image, mask)

    lsd = cv2.createLineSegmentDetector(cv2.LSD_REFINE_ADV)
    lines = lsd.detect(mask_no_boundary)[0]
    
    # Filter lines based on length
    arc_length_threshold = 0.1 * np.sqrt(cell_area)
    stress_fibers = []
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            length = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
            if length > 0:
                stress_fibers.append(line)
    return stress_fibers

def remove_cell_boundary(image, mask):
    """
    Remove the cell boundary using the actual image rather than the predicted mask.
    """
    # Convert the image to a NumPy array and ensure it is grayscale
    if isinstance(image, Image.Image):
        image = np.array(image)
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Apply thresholding to create a binary mask
    _, binary_mask = cv2.threshold(image, 10, 255, cv2.THRESH_BINARY)
    initial_mask = binary_mask.copy()
    contours, _ = cv2.findContours(initial_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Create a filled mask by drawing the contours
    filled_mask = np.zeros_like(initial_mask)
    cv2.drawContours(filled_mask, contours, -1, (255), thickness=cv2.FILLED)

    binary_mask = cv2.erode(filled_mask, np.ones((3, 3), np.uint8), iterations=20)

    mask[binary_mask == 0] = 0  # Set the cell boundary to zero in the mask

    return mask

def plot_histogram(areas, condition, folder_name):
    plt.figure(figsize=(10, 6))
    plt.hist(areas, bins=30, alpha=0.7, label=condition)
    plt.xlabel('Area')
    plt.ylabel('Frequency')
    plt.title(f'Histogram of Stress Fiber Areas for {condition} condition')
    plt.legend()
    plt.savefig(f'fast_drug_treatment/{folder_name}_stress_fiber_area_histogram_{condition}.png')
    plt.close()

with torch.no_grad():
    csv = [["file", "Stress Fibers", "Focal Adhesions", "Lamellipodia", "Filopodia"]]
    csv_to_save = [["Image Name", "Condition", "Stress Fibers", "Focal Adhesions", "Lamellipodia", "Filopodia"]]
    conditions = ["control", "rock"]

    folder_name = ""
    prediction_folder = os.path.join('fast_drug_treatment', 'predictions', folder_name)
    os.makedirs(prediction_folder, exist_ok=True)
    
    for i, image in enumerate(all_images):
        img = Image.open(image).convert("L")
        img = transform(img).unsqueeze(0).to(device)

        outputs = model(img)
        _, prediction_mask = torch.max(outputs, 1)
        prediction_mask = prediction_mask.cpu().numpy().squeeze()

        decoded_pred = decode_segmentation_masks(prediction_mask, colormap, 5)
        image_name = os.path.basename(image)
        image_condition = None
        stress_fiber_count = 0
        focal_adhesions_count = 0
        lamellipodia_count = 0
        filopodia_count = 0

        # Detect and count stress fibers
        stress_fiber_mask = (prediction_mask == 1).astype(np.uint8)
        cell_area = np.sum(stress_fiber_mask)
        
        stress_fiber_mask = decode_segmentation_masks((prediction_mask == 1).astype(np.uint8), colormap, 5) 
        
        stress_fiber_mask = cv2.cvtColor(stress_fiber_mask, cv2.COLOR_RGB2GRAY)
        stress_fiber_contours = detect_stress_fibers(Image.open(image).convert("L"), stress_fiber_mask, cell_area)
        stress_fiber_count = len(stress_fiber_contours)


        # Overlay stress fibers on the predicted mask
        overlay = decoded_pred.copy()
        for line in stress_fiber_contours:
            x1, y1, x2, y2 = line[0]
            cv2.line(overlay, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
        overlay_path = os.path.join(prediction_folder, f"overlay_{image_name}")
        Image.fromarray(overlay).save(overlay_path)

        # Count other structures
        pred_contours, _ = cv2.findContours((prediction_mask == 2).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        pred_contours = [cnt for cnt in pred_contours if cv2.contourArea(cnt) > 0]
        focal_adhesions_count = len(pred_contours)

        pred_contours, _ = cv2.findContours((prediction_mask == 3).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        pred_contours = [cnt for cnt in pred_contours if cv2.contourArea(cnt) > 0]
        lamellipodia_count = len(pred_contours)

        pred_contours, _ = cv2.findContours((prediction_mask == 4).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        pred_contours = [cnt for cnt in pred_contours if cv2.contourArea(cnt) > 0]
        filopodia_count = len(pred_contours)

        for condition in conditions:
            if condition in image_name:
                image_condition = condition

        filename = image_name.split("_")[0]
        csv.append([image_condition, stress_fiber_count, focal_adhesions_count, lamellipodia_count, filopodia_count])
        csv_to_save.append([image_name, image_condition, stress_fiber_count, focal_adhesions_count, lamellipodia_count, filopodia_count])

        # Save the decoded prediction
        prediction_path = os.path.join(prediction_folder, f"prediction_{image_name}")
        Image.fromarray(decoded_pred).save(prediction_path)

    df = pd.DataFrame(csv[1:], columns=csv[0])

    df_to_save = pd.DataFrame(csv_to_save[1:], columns=csv_to_save[0])
    df_to_save.to_csv(f'fast_drug_treatment/{folder_name}_counts.csv', index=False)

    df['file'] = pd.Categorical(df['file'], categories=conditions, ordered=True)
    df = df.sort_values('file')

    df_melted = pd.melt(df, id_vars=['file'], var_name='Protein', value_name='Percentage')

    # Create box plot
    plt.figure(figsize=(16, 8))
    ax = sns.boxplot(x='file', y='Percentage', hue='Protein', data=df_melted, dodge=True)
    plt.xlabel('Condition')
    plt.ylabel('Count')
    plt.title(f'Box Plot of distribution of actin substructure counts across drug conditions for {folder_name}')
    plt.legend(title='Actin substructure')

    # Perform statistical tests and add annotations
    proteins = df_melted['Protein'].unique()
    pairs = [((condition, protein), ("control", protein)) for condition in conditions if condition != "control" for protein in proteins]

    for i, (condition, protein) in enumerate(pairs):
        data1 = df_melted[(df_melted['file'] == condition[0]) & (df_melted['Protein'] == condition[1])]['Percentage']
        data2 = df_melted[(df_melted['file'] == "control") & (df_melted['Protein'] == condition[1])]['Percentage']
        stat, p = mannwhitneyu(data1, data2)
        y_max = max(data1.max(), data2.max())
        x_offset = ((i % 4)-1) * 0.2  # Adjust x-offset for better visibility
        ax.text(x=conditions.index(condition[0]) + x_offset, y=y_max + 1, s=f'p={p:.3e}', ha='center')

    plt.savefig(f'fast_drug_treatment/{folder_name}_boxplot_counts.png')
    plt.close()